# OOV: word2vec

Counts OOV and lookup metrics of `word2vec-google-news-300` on the ART tokens. The tokens come from the shared `preprocess()` in `preprocessing.py` (edit it there); the lookup is the one of the static solutions. Results are written to `results/w2v.json`; `summary.ipynb` runs all three evals and compares them.

In [1]:
%pip install -q datasets gensim nltk


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2

from collections import Counter

import gensim.downloader as api
import nltk
from datasets import load_dataset

from preprocessing import preprocess  # shared by all three evals, edit preprocessing.py

nltk.download("punkt_tab", quiet=True)

True

In [3]:
EMBEDDINGS = {
    "w2v": "word2vec-google-news-300",
}

In [4]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 16801, valid: 1000, test: 1532


In [5]:
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token for token in tokens if any(character.isalnum() for character in token)]

In [6]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1
    return processed


def preprocess_split(split):
    # no cache: datasets would not notice an edit in preprocessing.py and reuse the old tokens
    return split.map(preprocess_example, load_from_cache_file=False)

In [7]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

Map:   0%|          | 0/16801 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1532 [00:00<?, ? examples/s]

In [8]:
embeddings = {name: api.load(model_id) for name, model_id in EMBEDDINGS.items()}

for name, kv in embeddings.items():
    print(f"{name}: {len(kv.index_to_key)} words, {kv.vector_size} dimensions")

w2v: 3000000 words, 300 dimensions


### Lookup

In [9]:
sentences = [sentence for split in (train_tokens, valid_tokens, test_tokens) for field in TEXT_FIELDS for sentence in split[field]]
token_counts = Counter(token for sentence in sentences for token in sentence)
total_tokens = sum(token_counts.values())

print(f"sentences: {len(sentences)}, tokens: {total_tokens}, types: {len(token_counts)}")


def lookup(kv, token):
    if token in kv:
        return "exact"
    if token.lower() in kv:
        return "lowercase"
    return "oov"

sentences: 77332, tokens: 670395, types: 19041


In [10]:
metrics = {
    "oov/tokens": total_tokens,
    "oov/types": len(token_counts),
}
oov_types = {}

for name, kv in embeddings.items():
    status = {token: lookup(kv, token) for token in token_counts}
    occurrences = Counter()
    for token, count in token_counts.items():
        occurrences[status[token]] += count
    oov_types[name] = {token for token, result in status.items() if result == "oov"}

    metrics.update({
        f"oov/{name}/vocab_size": len(kv.index_to_key),
        f"oov/{name}/hits_exact": occurrences["exact"],
        f"oov/{name}/hits_lowercase": occurrences["lowercase"],
        f"oov/{name}/occurrences": occurrences["oov"],
        f"oov/{name}/rate": occurrences["oov"] / total_tokens,
        f"oov/{name}/unique_types": len(oov_types[name]),
        f"oov/{name}/type_rate": len(oov_types[name]) / len(token_counts),
        f"oov/{name}/empty_sentences": sum(all(status[token] == "oov" for token in sentence) for sentence in sentences),
    })

for key, value in metrics.items():
    print(f"{key:32} {value:.4f}" if isinstance(value, float) else f"{key:32} {value}")

oov/tokens                       670395
oov/types                        19041
oov/w2v/vocab_size               3000000
oov/w2v/hits_exact               591672
oov/w2v/hits_lowercase           18
oov/w2v/occurrences              78705
oov/w2v/rate                     0.1174
oov/w2v/unique_types             1142
oov/w2v/type_rate                0.0600
oov/w2v/empty_sentences          6


In [11]:
for name, types in oov_types.items():
    top = sorted(types, key=token_counts.get, reverse=True)[:20]
    print(f"{name}: {[(token, token_counts[token]) for token in top]}")

w2v: [('to', 26514), ('a', 22768), ('and', 15023), ('of', 6595), ("'s", 5582), ('20', 75), ('10', 72), ('didnt', 58), ('cancelled', 48), ('100', 44), ('50', 40), ('30', 39), ('co-worker', 30), ('15', 25), ('12', 24), ('18', 21), ('16', 17), ('200', 16), ('co-workers', 13), ('21', 11)]


In [12]:
import json
from pathlib import Path

# picked up by summary.ipynb
Path("results").mkdir(exist_ok=True)
for name in embeddings:
    Path(f"results/{name}.json").write_text(json.dumps({
        "model": EMBEDDINGS[name],
        "metrics": {k.split("/")[-1]: v for k, v in metrics.items() if k.startswith(f"oov/{name}/") or k in ("oov/tokens", "oov/types")},
        "top_oov": [(token, token_counts[token]) for token in sorted(oov_types[name], key=token_counts.get, reverse=True)[:20]],
    }, indent=1))